<a href="https://colab.research.google.com/github/Ezhreal/FormationIA/blob/incourse/exercicios/statsmodels_sklearn_workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# CENÁRIO: Predição de Preços de Imóveis na Califórnia
# Demonstração de análise estatística com Statsmodels e implementação para produção com Scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import warnings
warnings.filterwarnings('ignore')  # Ignorar warnings para melhor visualização

# Carregando o dataset California Housing
california = fetch_california_housing()
X = pd.DataFrame(california.data, columns=california.feature_names)
y = california.target

# Criando DataFrame para melhor manipulação
df = pd.DataFrame(california.data, columns=california.feature_names)
df['MedHouseVal'] = california.target  # Adicionando a variável alvo (preço médio)

print("=== DATASET CALIFORNIA HOUSING ===")
print(f"Número de amostras: {df.shape[0]}")
print(f"Número de características: {df.shape[1] - 1}")  # -1 para excluir a variável alvo
print("\nInformações sobre as features:")
for i, feature in enumerate(california.feature_names):
    print(f"  {feature}: {california.feature_names[i]}")
print("\nVariável alvo: MedHouseVal (Valor mediano das casas em $100,000s)")
print("\nPrimeiras linhas do dataset:")
print(df.head())
print("\nEstatísticas descritivas:")
print(df.describe())

=== DATASET CALIFORNIA HOUSING ===
Número de amostras: 20640
Número de características: 8

Informações sobre as features:
  MedInc: MedInc
  HouseAge: HouseAge
  AveRooms: AveRooms
  AveBedrms: AveBedrms
  Population: Population
  AveOccup: AveOccup
  Latitude: Latitude
  Longitude: Longitude

Variável alvo: MedHouseVal (Valor mediano das casas em $100,000s)

Primeiras linhas do dataset:
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  MedHouseVal  
0    -122.23        4.526  
1    -122.22        3.585  
2    -122.24        3.521  
3    -122.25        3.413  
4    -122.25

In [3]:
print("\n" + "="*70)
print("FASE 1: ANÁLISE EXPLORATÓRIA E ESTATÍSTICA COM STATSMODELS")
print("="*70)

# Verificando correlações com a variável alvo
correlacoes = df.corr()
print("\nCorrelações com a variável alvo (MedHouseVal):")
print(correlacoes['MedHouseVal'].sort_values(ascending=False))

# Visualização de correlações
plt.figure(figsize=(12, 10))
sns.heatmap(correlacoes, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Matriz de Correlação - California Housing')
plt.savefig('california_correlacao.png')
plt.close()

# Criando matriz de scatter plots para variáveis mais correlacionadas
# Encontrando as 5 variáveis mais correlacionadas com a variável alvo
top_correlations = correlacoes['MedHouseVal'].abs().sort_values(ascending=False)[1:6]  # [1:6] para excluir a própria variável alvo
variaveis_importantes = top_correlations.index.tolist() + ['MedHouseVal']

plt.figure(figsize=(12, 10))
sns.pairplot(df[variaveis_importantes])
plt.savefig('california_pairplot.png')
plt.close()

# Adicionando constante para o intercepto no statsmodels
X_sm = sm.add_constant(X)

# Verificando multicolinearidade com VIF (Variance Inflation Factor)
vif_data = pd.DataFrame()
vif_data["Variável"] = X_sm.columns
vif_data["VIF"] = [variance_inflation_factor(X_sm.values, i) for i in range(X_sm.shape[1])]
print("\nFator de Inflação da Variância (VIF):")
print(vif_data.sort_values('VIF', ascending=False))

# Modelo Statsmodels
modelo_sm = sm.OLS(y, X_sm).fit()
print("\nResumo estatístico do modelo:")
print(modelo_sm.summary())

# Análise dos resíduos
residuos = modelo_sm.resid
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.scatter(modelo_sm.fittedvalues, residuos)
plt.axhline(y=0, color='r', linestyle='--')
plt.title('Resíduos vs Valores Ajustados')
plt.xlabel('Valores Ajustados')
plt.ylabel('Resíduos')

plt.subplot(1, 2, 2)
plt.hist(residuos, bins=30)
plt.title('Distribuição dos Resíduos')
plt.xlabel('Resíduos')
plt.savefig('california_residuos.png')
plt.close()

# Teste de normalidade usando Shapiro-Wilk
print("\nTeste Shapiro-Wilk (normalidade dos resíduos):")
shapiro_test = stats.shapiro(residuos)
print(f"Estatística: {shapiro_test[0]:.4f}")
print(f"p-valor: {shapiro_test[1]:.4f}")
if shapiro_test[1] < 0.05:
    print("Os resíduos não seguem uma distribuição normal (p < 0.05)")
else:
    print("Os resíduos seguem uma distribuição normal (p > 0.05)")

# Para o gráfico Q-Q (complemento visual do teste de normalidade)
plt.figure(figsize=(10, 6))
stats.probplot(residuos, plot=plt)
plt.title('Gráfico Q-Q dos Resíduos')
plt.savefig('california_qq_plot.png')
plt.close()

print("\nTeste Breusch-Pagan (homocedasticidade):")
bp_test = sm.stats.diagnostic.het_breuschpagan(residuos, X_sm)
print(f"Estatística: {bp_test[0]:.4f}")
print(f"p-valor: {bp_test[1]:.4f}")
if bp_test[1] < 0.05:
    print("Presença de heterocedasticidade (p < 0.05)")
else:
    print("Homocedasticidade (p > 0.05)")

# Análise dos resíduos padronizados para detectar outliers
residuos_padronizados = modelo_sm.get_influence().resid_studentized_internal
plt.figure(figsize=(10, 6))
plt.scatter(range(len(residuos_padronizados)), residuos_padronizados)
plt.axhline(y=3, color='r', linestyle='--')
plt.axhline(y=-3, color='r', linestyle='--')
plt.title('Resíduos Padronizados')
plt.ylabel('Resíduo Studentizado')
plt.xlabel('Observação')
plt.savefig('california_residuos_padronizados.png')
plt.close()


FASE 1: ANÁLISE EXPLORATÓRIA E ESTATÍSTICA COM STATSMODELS

Correlações com a variável alvo (MedHouseVal):
MedHouseVal    1.000000
MedInc         0.688075
AveRooms       0.151948
HouseAge       0.105623
AveOccup      -0.023737
Population    -0.024650
Longitude     -0.045967
AveBedrms     -0.046701
Latitude      -0.144160
Name: MedHouseVal, dtype: float64

Fator de Inflação da Variância (VIF):
     Variável           VIF
0       const  17082.623698
7    Latitude      9.297624
8   Longitude      8.962263
3    AveRooms      8.342786
4   AveBedrms      6.994995
1      MedInc      2.501295
2    HouseAge      1.241254
5  Population      1.138125
6    AveOccup      1.008324

Resumo estatístico do modelo:
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.606
Model:                            OLS   Adj. R-squared:                  0.606
Method:                 Least Squares   F-statistic:   

<Figure size 1200x1000 with 0 Axes>

In [4]:
# Transformação logarítmica da variável alvo
y_log = np.log(y)

# Adicionando constante para o intercepto
X_sm_log = sm.add_constant(X)

# Modelo com variável transformada
modelo_sm_log = sm.OLS(y_log, X_sm_log).fit()
print("\nResumo estatístico do modelo após transformação logarítmica:")
print(modelo_sm_log.summary())

# Verificar novamente a normalidade dos resíduos
residuos_log = modelo_sm_log.resid
shapiro_test_log = stats.shapiro(residuos_log)
print("\nTeste Shapiro-Wilk após transformação logarítmica:")
print(f"Estatística: {shapiro_test_log[0]:.4f}")
print(f"p-valor: {shapiro_test_log[1]:.4f}")


Resumo estatístico do modelo após transformação logarítmica:
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.614
Model:                            OLS   Adj. R-squared:                  0.614
Method:                 Least Squares   F-statistic:                     4109.
Date:                Sat, 26 Apr 2025   Prob (F-statistic):               0.00
Time:                        13:36:06   Log-Likelihood:                -7819.3
No. Observations:               20640   AIC:                         1.566e+04
Df Residuals:                   20631   BIC:                         1.573e+04
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------

In [9]:
# Dividindo os dados em treino e teste
from sklearn.model_selection import train_test_split

# Usando o X e y que já temos, mas aplicando a transformação logarítmica na variável alvo
y_log = np.log(y)
X_train, X_test, y_train_log, y_test_log = train_test_split(X, y_log, test_size=0.2, random_state=42)

# Agora podemos treinar o modelo Ridge
from sklearn.linear_model import Ridge

# Criar e treinar modelo Ridge
ridge_model = Ridge(alpha=1.0)  # alpha é o parâmetro de regularização
ridge_model.fit(X_train, y_train_log)

# Avaliar o modelo
from sklearn.metrics import mean_squared_error, r2_score
y_pred_log = ridge_model.predict(X_test)
r2 = r2_score(y_test_log, y_pred_log)
rmse = np.sqrt(mean_squared_error(y_test_log, y_pred_log))

print(f"R² do modelo Ridge: {r2:.4f}")
print(f"RMSE do modelo Ridge (em log): {rmse:.4f}")

# Se quiser ver os resultados em escala original (não logarítmica)
y_pred_original = np.exp(y_pred_log)
y_test_original = np.exp(y_test_log)
rmse_original = np.sqrt(mean_squared_error(y_test_original, y_pred_original))

print(f"RMSE do modelo Ridge (escala original): {rmse_original:.4f}")

R² do modelo Ridge: 0.5940
RMSE do modelo Ridge (em log): 0.3630
RMSE do modelo Ridge (escala original): 1.5247


In [10]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0))
])

# Treinando o pipeline com os dados logarítmicos
pipeline.fit(X_train, y_train_log)

# Salvando o modelo para produção
import joblib
joblib.dump(pipeline, 'modelo_california_ridge_log.pkl')

['modelo_california_ridge_log.pkl']

In [13]:
# Assegurando que temos o pipeline salvo
pipeline.fit(X_train, y_train_log)
joblib.dump(pipeline, 'modelo_california_ridge_log.pkl')

def prever_preco_imovel(features):
    """
    Prevê o preço de um imóvel com base nas características fornecidas.

    Args:
        features (dict): Dicionário com as características do imóvel

    Returns:
        float: Preço previsto em $100,000
    """
    modelo_prod = joblib.load('modelo_california_ridge_log.pkl')
    X_novo = pd.DataFrame([features])

    # Prever no espaço logarítmico e converter de volta
    preco_log = modelo_prod.predict(X_novo)[0]
    preco = np.exp(preco_log)

    return preco

# Criando exemplos de imóveis com características diversas

# Imóvel em área de alta renda próximo à costa (San Francisco)
imovel_luxo = {
    'MedInc': 12.5,           # Renda mediana alta
    'HouseAge': 35.0,         # Casa mais antiga/estabelecida
    'AveRooms': 7.5,          # Mais cômodos que a média
    'AveBedrms': 2.5,         # Número médio-alto de quartos
    'Population': 1200.0,      # População moderada
    'AveOccup': 2.1,          # Média de ocupantes
    'Latitude': 37.75,        # São Francisco
    'Longitude': -122.45      # São Francisco
}

# Imóvel em área suburbana de renda média
imovel_suburbio = {
    'MedInc': 6.2,            # Renda mediana média
    'HouseAge': 15.0,         # Casa mais nova
    'AveRooms': 6.0,          # Número médio de cômodos
    'AveBedrms': 2.0,         # Número médio de quartos
    'Population': 2500.0,      # População suburbana
    'AveOccup': 2.8,          # Mais ocupantes
    'Latitude': 37.35,        # Sul da Baía
    'Longitude': -122.0       # Sul da Baía
}

# Imóvel em área rural/interior de baixa renda
imovel_rural = {
    'MedInc': 3.0,            # Renda mediana baixa
    'HouseAge': 45.0,         # Casa mais antiga
    'AveRooms': 5.2,          # Menos cômodos
    'AveBedrms': 1.8,         # Menos quartos
    'Population': 800.0,       # População menor
    'AveOccup': 3.2,          # Mais ocupantes por casa
    'Latitude': 36.5,         # Interior da Califórnia
    'Longitude': -119.5       # Interior da Califórnia
}

# Testando o modelo com os exemplos
preco_luxo = prever_preco_imovel(imovel_luxo)
preco_suburbio = prever_preco_imovel(imovel_suburbio)
preco_rural = prever_preco_imovel(imovel_rural)

print("\nPrevisões de preços para diferentes tipos de imóveis:")
print(f"Imóvel de luxo em área de alta renda: ${preco_luxo*100000:.2f}")
print(f"Imóvel em área suburbana: ${preco_suburbio*100000:.2f}")
print(f"Imóvel em área rural/interior: ${preco_rural*100000:.2f}")


Previsões de preços para diferentes tipos de imóveis:
Imóvel de luxo em área de alta renda: $1610380.65
Imóvel em área suburbana: $420147.04
Imóvel em área rural/interior: $146840.91


In [15]:
# Modelo sem a variável 'Population' que não é significativa
X_sm_reduzido = X_sm.drop('Population', axis=1)
modelo_sm_reduzido = sm.OLS(y, X_sm_reduzido).fit()
print("\nResumo do modelo sem 'Population':")
print(modelo_sm_reduzido.summary())

# Ou testar um modelo apenas com as variáveis de maior correlação
# MedInc, AveRooms, HouseAge, Latitude
variaveis_top = ['MedInc', 'AveRooms', 'HouseAge', 'Latitude']
X_sm_top = sm.add_constant(X[variaveis_top])
modelo_sm_top = sm.OLS(y, X_sm_top).fit()
print("\nResumo do modelo com apenas as variáveis de maior correlação:")
print(modelo_sm_top.summary())


Resumo do modelo sem 'Population':
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.606
Model:                            OLS   Adj. R-squared:                  0.606
Method:                 Least Squares   F-statistic:                     4538.
Date:                Sat, 26 Apr 2025   Prob (F-statistic):               0.00
Time:                        13:50:29   Log-Likelihood:                -22624.
No. Observations:               20640   AIC:                         4.526e+04
Df Residuals:                   20632   BIC:                         4.533e+04
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        -36